# Construccion del catalogo de zonas OPSD


Este notebook descarga la metadata de Open Power System Data desde `https://data.open-power-system-data.org/time_series/2020-10-06/datapackage.json`, extrae las zonas disponibles para la variable de demanda electrica real horaria y genera el catalogo versionado `../energy-forecast-app/data/reference/opsd_zones.csv` usado por la aplicacion.


Importa las librerias necesarias para leer JSON, trabajar con rutas, extraer patrones de texto y exportar el catalogo como CSV.


In [1]:
import json
import re
from urllib.request import urlopen
from pathlib import Path

import pandas as pd


Define la URL de metadata, la ruta de salida local y la variable objetivo usada para construir el catalogo de zonas.


In [2]:
PROJECT_ROOT = Path.cwd().parent
APP_ROOT = PROJECT_ROOT / "energy-forecast-app"
DATAPACKAGE_URL = "https://data.open-power-system-data.org/time_series/2020-10-06/datapackage.json"
OUTPUT_PATH = APP_ROOT / "data" / "reference" / "opsd_zones.csv"
RESOURCE_PATH = "time_series_60min_singleindex.csv"
TARGET_VARIABLE = "load_actual_entsoe_transparency"


Estas funciones encapsulan la descarga de metadata, la seleccion del recurso horario y la normalizacion de nombres y tipos de zona desde la descripcion de OPSD.


In [3]:
def load_datapackage(url: str) -> dict:
    with urlopen(url) as response:
        return json.load(response)

def find_resource(datapackage: dict, resource_path: str) -> dict:
    for resource in datapackage.get("resources", []):
        if resource.get("path") == resource_path:
            return resource
    raise ValueError(f"No se encontro el recurso {resource_path}")


def zone_name_from_description(description: str) -> str:
    name = re.sub(r"^Total load in ", "", description)
    name = re.sub(r" in MW.*$", "", name)
    return re.sub(r"\s*\((control area|bidding zone)\)\s*", "", name).strip()


def zone_kind_from_description(description: str) -> str:
    if "(control area)" in description:
        return "control_area"
    if "(bidding zone)" in description:
        return "bidding_zone"
    if description.startswith("Total load in "):
        return "country"
    return "unknown"


Construye el DataFrame del catalogo usando `opsdProperties.Region` como llave confiable y filtrando solo la variable de demanda real publicada por ENTSO-E.


In [4]:
datapackage = load_datapackage(DATAPACKAGE_URL)
resource = find_resource(datapackage, RESOURCE_PATH)

rows = []
for field in resource.get("schema", {}).get("fields", []):
    properties = field.get("opsdProperties", {})
    if properties.get("Variable") != TARGET_VARIABLE:
        continue

    description = field.get("description", "")
    rows.append(
        {
            "code": properties.get("Region", ""),
            "name": zone_name_from_description(description),
            "kind": zone_kind_from_description(description),
            "source_variable": properties.get("Variable", ""),
            "source_description": description,
        }
    )

zones = pd.DataFrame(rows).drop_duplicates(subset=["code"]).sort_values("code")
zones


,code,name,kind,source_variable,source_description
0,AT,Austria,country,load_actual_entsoe_transparency,Total load in Austria in MW as published on EN...
1,BE,Belgium,country,load_actual_entsoe_transparency,Total load in Belgium in MW as published on EN...
2,BG,Bulgaria,country,load_actual_entsoe_transparency,Total load in Bulgaria in MW as published on E...
3,CH,Switzerland,country,load_actual_entsoe_transparency,Total load in Switzerland in MW as published o...
4,CY,Cyprus,country,load_actual_entsoe_transparency,Total load in Cyprus in MW as published on ENT...
5,CZ,Czech Republic,country,load_actual_entsoe_transparency,Total load in Czech Republic in MW as publishe...
6,DE,Germany,country,load_actual_entsoe_transparency,Total load in Germany in MW as published on EN...
7,DE_50hertz,50Hertz,control_area,load_actual_entsoe_transparency,Total load in 50Hertz (control area) in MW as ...
8,DE_LU,DE-LU,bidding_zone,load_actual_entsoe_transparency,Total load in DE-LU (bidding zone) in MW as pu...
9,DE_amprion,Amprion,control_area,load_actual_entsoe_transparency,Total load in Amprion (control area) in MW as ...


Valida que no existan codigos vacios o duplicados antes de exportar el catalogo final.


In [5]:
if zones.empty:
    raise ValueError("No se encontraron zonas para la variable objetivo.")
if zones["code"].eq("").any():
    raise ValueError("Existen zonas sin codigo.")
if zones["code"].duplicated().any():
    raise ValueError("Existen codigos de zona duplicados.")

zones["kind"].value_counts()


kind
country         34
bidding_zone    19
control_area     4
Name: count, dtype: int64

Exporta el catalogo versionado que sera consumido por la aplicacion de escritorio.


In [7]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
zones.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH.relative_to(APP_ROOT)


WindowsPath('data/reference/opsd_zones.csv')